 ### Setting up the Connection w/ QCloud

In [ ]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url="",
    api_key="",
)

### Sample data

In [ ]:
documents = [
    {"title": "Intro to Feature Scaling", "text": "Feature scaling is used in ML to normalize data."},
    {"title": "Why Feature Scaling Matters", "text": "It helps improve model performance and training speed."},
    {"title": "Gradient Descent", "text": "Scaling improves convergence of gradient descent."},
    {"title": "Standardization Explained", "text": "Standardization ensures features have mean 0 and unit variance."},
    {"title": "K-Means Clustering", "text": "Without scaling, K-means is biased toward features with larger values."},
    {"title": "DBSCAN Tips", "text": "Normalized input boosts DBSCAN's performance."},
    {"title": "Neural Networks", "text": "Neural networks converge faster with scaled input data."},
    {"title": "Detecting Outliers", "text": "Robust scaling reduces impact of extreme values."},
    {"title": "Feature Engineering", "text": "Scaling is a crucial step before feeding data into models."},
    {"title": "PCA Overview", "text": "PCA assumes data is centered and scaled."},
    {"title": "Time Series Forecasting", "text": "Scaling stabilizes dynamic input values over time."},
    {"title": "Choosing Scalers", "text": "MinMaxScaler and StandardScaler suit different scenarios."},
    {"title": "Data Preprocessing", "text": "Scaling is part of almost every ML pipeline."},
    {"title": "SVM Accuracy", "text": "SVMs require scaled input for optimal margin calculations."},
]


### Load Embedding Models

In [ ]:
from fastembed import TextEmbedding, LateInteractionTextEmbedding, SparseTextEmbedding

dense_embedding_model = TextEmbedding("sentence-transformers/all-MiniLM-L6-v2")
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25")
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0") ## Used for re-ranking

### Extract Text -> Embed

In [ ]:
texts = [doc["text"] for doc in documents]

# Embed
dense_embeddings = list(dense_embedding_model.embed(texts))
bm25_embeddings = list(bm25_embedding_model.embed(texts))
late_interaction_embeddings = list(late_interaction_embedding_model.embed(texts))

### Create a new collection to support all 3 vectors

In [ ]:
from qdrant_client.models import models

qdrant_client.create_collection(
    "hybrid-search",
    vectors_config={
        "all-MiniLM-L6-v2": models.VectorParams(
            size=384,
            distance=models.Distance.COSINE,
        ),
        "colbertv2.0": models.VectorParams(
            size=128,
            distance=models.Distance.COSINE,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            )
        ),
    },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF
        )
    }
)

### Upsert

In [ ]:
from qdrant_client.models import PointStruct
points = []
for idx, (dense_embedding, bm25_embedding, late_interaction_embedding, doc) in enumerate(zip(dense_embeddings, bm25_embeddings, late_interaction_embeddings, documents)):

    point = PointStruct(
        id=idx,
        vector={
            "all-MiniLM-L6-v2": dense_embedding,
            "bm25": bm25_embedding.as_object(),
            "colbertv2.0": late_interaction_embedding,
        },
        payload={"document": doc}
    )
    points.append(point)

operation_info = qdrant_client.upsert(
    collection_name="hybrid-search",
    points=points
)

### Hybdrid Search + Re-rank

In [40]:
from qdrant_client.models import Prefetch, SparseVector

query_text = "why does scaling make gradient descent more efficient?"

dense_query_vector = list(dense_embedding_model.embed([query_text]))[0]
sparse_query_vector = list(bm25_embedding_model.embed([query_text]))[0]
late_query_vector = list(late_interaction_embedding_model.embed([query_text]))[0]

prefetch = [
    Prefetch(
        query=dense_query_vector,
        using="all-MiniLM-L6-v2",
        limit=10,
    ),
    Prefetch(
        query=SparseVector(**sparse_query_vector.as_object()),
        using="bm25",
        limit=10,
    )
]


results = qdrant_client.query_points(
    collection_name="hybrid-search",
    prefetch=prefetch,
    query=late_query_vector,
    using="colbertv2.0",
    limit=5,
    with_payload=True
)


for result in results.points:
    print(f"Score: {result.score:.4f}")
    print("Title:", result.payload["document"]["title"])
    print("Text:", result.payload["document"]["text"])
    print("-" * 10)

Score: 7.9885
Title: Gradient Descent
Text: Scaling improves convergence of gradient descent.
----------
Score: 5.7610
Title: Time Series Forecasting
Text: Scaling stabilizes dynamic input values over time.
----------
Score: 5.3289
Title: Detecting Outliers
Text: Robust scaling reduces impact of extreme values.
----------
Score: 5.2712
Title: Feature Engineering
Text: Scaling is a crucial step before feeding data into models.
----------
Score: 5.2637
Title: Neural Networks
Text: Neural networks converge faster with scaled input data.
----------


### Hybrid Search

In [39]:
from qdrant_client.models import  SparseVector, NamedVector

query_text = "why does scaling make gradient descent more efficient?"
prefetch = [
    Prefetch(
        query=dense_query_vector,
        using="all-MiniLM-L6-v2",
        limit=10,
    ),
    Prefetch(
        query=SparseVector(**sparse_query_vector.as_object()),
        using="bm25",
        limit=10,
    )
]
results = qdrant_client.query_points(
    collection_name="hybrid-search",
    prefetch=prefetch,
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=5,
    with_payload=True
)
# print(results)
for result in results.points:
    print(f"Score: {result.score:.4f}")
    print("Title:", result.payload["document"]["title"])
    print("Text:", result.payload["document"]["text"])
    print("-" * 10)


## External re rank call
# document_list = [point.payload['document'] for point in results.points]
# rerank_results = co.rerank(
#     model="rerank-english-v3.0",
#     query=query_text,
#     documents=document_list,
#     top_n=5,
# )
#



Score: 1.0000
Title: Gradient Descent
Text: Scaling improves convergence of gradient descent.
----------
Score: 0.4333
Title: Neural Networks
Text: Neural networks converge faster with scaled input data.
----------
Score: 0.3750
Title: Detecting Outliers
Text: Robust scaling reduces impact of extreme values.
----------
Score: 0.3750
Title: Data Preprocessing
Text: Scaling is part of almost every ML pipeline.
----------
Score: 0.3667
Title: Time Series Forecasting
Text: Scaling stabilizes dynamic input values over time.
----------
